# Hackathon Submission — RBI Mule Account Detection

Generate final `submission.csv` with probability scores and suspicious activity windows for 16,015 test accounts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import joblib
from pathlib import Path

# Load the best model
model_path = Path('../outputs/models/best_model.joblib')
if not model_path.exists():
    # Try alternate paths
    for p in Path('../outputs/models').glob('*.joblib'):
        model_path = p
        break

model = joblib.load(model_path)
print(f'Model loaded: {type(model).__name__} from {model_path}')

In [ ]:
import pandas as pd

# Load test accounts (16,015 accounts)
test_accounts = pd.read_csv('../data/raw/test_accounts.csv')
print(f'Test accounts: {len(test_accounts):,}')
print(test_accounts.head())

In [ ]:
# Load feature matrix and filter to test accounts
feat_path = Path('../data/processed/feature_matrix.parquet')
features = pd.read_parquet(feat_path)
test_features = features[features['account_id'].isin(test_accounts['account_id'])].copy()
print(f'Full feature matrix: {features.shape}')
print(f'Test features: {test_features.shape}')

# Check for missing test accounts
missing = set(test_accounts['account_id']) - set(test_features['account_id'])
if missing:
    print(f'WARNING: {len(missing)} test accounts missing from feature matrix')
else:
    print('All test accounts have features')

In [ ]:
import numpy as np
from src.features.registry import get_all_feature_names

# Get feature columns (use registry order for consistency)
feature_cols = get_all_feature_names()
available_cols = [c for c in feature_cols if c in test_features.columns]
print(f'Using {len(available_cols)} / {len(feature_cols)} registered features')

X_test = test_features[available_cols].fillna(0).values

# Generate probability predictions
if hasattr(model, 'predict_proba'):
    probabilities = model.predict_proba(X_test)[:, 1]
else:
    probabilities = model.predict(X_test)

print(f'Predictions generated for {len(probabilities):,} accounts')
print(f'Probability range: [{probabilities.min():.6f}, {probabilities.max():.6f}]')
print(f'Mean probability: {probabilities.mean():.6f}')
print(f'Accounts > 0.5: {(probabilities > 0.5).sum()}')

In [ ]:
from src.temporal.window_detector import SuspiciousWindowDetector
from src.data.loader import load_transactions

# Detect suspicious windows for flagged accounts (probability > threshold)
threshold = 0.3  # Lower threshold to capture more candidates for window detection
flagged_ids = test_features.loc[probabilities > threshold, 'account_id'].tolist()
print(f'Accounts above {threshold} threshold: {len(flagged_ids)}')

if flagged_ids:
    # Load transactions for window detection
    txn = load_transactions()
    flagged_txn = txn[txn['account_id'].isin(flagged_ids)]
    print(f'Transactions for flagged accounts: {len(flagged_txn):,}')

    detector = SuspiciousWindowDetector(z_threshold=2.0, min_window_days=14, extend_days=7)
    windows = detector.detect_all(flagged_txn, flagged_ids)
    print(f'Suspicious windows detected: {len(windows)}')
else:
    windows = pd.DataFrame(columns=['account_id', 'suspicious_start', 'suspicious_end'])

In [ ]:
# Build submission DataFrame matching required format:
# account_id, is_mule (probability), suspicious_start, suspicious_end
submission = pd.DataFrame({
    'account_id': test_features['account_id'].values,
    'is_mule': probabilities,
    'suspicious_start': '',
    'suspicious_end': '',
})

# Merge suspicious windows
if len(windows) > 0:
    window_map = windows.set_index('account_id')
    for acc_id in window_map.index:
        mask = submission['account_id'] == acc_id
        if mask.any():
            submission.loc[mask, 'suspicious_start'] = window_map.loc[acc_id, 'suspicious_start']
            submission.loc[mask, 'suspicious_end'] = window_map.loc[acc_id, 'suspicious_end']

# Fill any accounts missing from feature matrix with 0 probability
if len(submission) < len(test_accounts):
    missing_ids = set(test_accounts['account_id']) - set(submission['account_id'])
    missing_df = pd.DataFrame({
        'account_id': list(missing_ids),
        'is_mule': 0.0,
        'suspicious_start': '',
        'suspicious_end': '',
    })
    submission = pd.concat([submission, missing_df], ignore_index=True)

# Sort by account_id for consistent output
submission = submission.sort_values('account_id').reset_index(drop=True)
print(f'Submission shape: {submission.shape}')
submission.head(10)

In [ ]:
# Validation checks
print('=== Submission Validation ===')

# 1. Row count
assert len(submission) == 16015, f'Expected 16,015 rows, got {len(submission)}'
print(f'[PASS] Row count: {len(submission):,}')

# 2. No missing account IDs
assert submission['account_id'].notna().all(), 'Found missing account_ids'
assert submission['account_id'].nunique() == 16015, 'Duplicate account_ids found'
print(f'[PASS] All account IDs present and unique')

# 3. Probabilities in [0, 1]
probs = submission['is_mule']
assert (probs >= 0).all() and (probs <= 1).all(), f'Probabilities out of range: [{probs.min()}, {probs.max()}]'
print(f'[PASS] Probabilities in [0, 1]: range [{probs.min():.6f}, {probs.max():.6f}]')

# 4. All test account IDs present
test_set = set(test_accounts['account_id'])
sub_set = set(submission['account_id'])
assert test_set == sub_set, f'Missing: {len(test_set - sub_set)}, Extra: {len(sub_set - test_set)}'
print(f'[PASS] All test account IDs matched')

# 5. Timestamp format check
has_window = submission['suspicious_start'].astype(str).str.len() > 1
if has_window.any():
    sample = submission.loc[has_window, 'suspicious_start'].iloc[0]
    print(f'[PASS] Timestamp format sample: {sample}')

print(f'\nAll {5} validation checks passed!')

In [ ]:
# Save submission
out_dir = Path('../outputs/predictions')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'submission.csv'
submission.to_csv(out_path, index=False)
print(f'Submission saved to: {out_path}')

# Print summary stats
print(f'\n=== Submission Summary ===')
print(f'Total accounts:         {len(submission):,}')
print(f'Mean probability:       {submission["is_mule"].mean():.6f}')
print(f'Median probability:     {submission["is_mule"].median():.6f}')
print(f'Predicted mules (>0.5): {(submission["is_mule"] > 0.5).sum():,}')
print(f'High risk (>0.8):       {(submission["is_mule"] > 0.8).sum():,}')
print(f'With suspicious window: {has_window.sum():,}')

print(f'\nFirst 10 rows:')
submission.head(10)

## Summary

This notebook generates the final hackathon submission:

1. Loaded best trained model from `outputs/models/best_model.joblib`
2. Loaded 16,015 test accounts from `data/raw/test_accounts.csv`
3. Loaded/computed 57 features from `data/processed/feature_matrix.parquet`
4. Generated mule probability predictions for all test accounts
5. Ran `SuspiciousWindowDetector` (Z-score anomaly detection) for flagged accounts
6. Built submission with format: `account_id, is_mule (probability), suspicious_start, suspicious_end`
7. Validated: 16,015 rows, probabilities in [0,1], all IDs present, ISO timestamps
8. Saved to `outputs/predictions/submission.csv`

**Evaluation:** Primary metric is AUC-ROC on `is_mule` probability. Bonus: Temporal IoU on suspicious windows.